# Vector Database with ChromaDB: embeddings, search and RAG

In this notebook we follow one idea in four steps:

1. **Meaning as numbers:** turn sentences into vectors and measure how close they are.
2. **A small vector database:** store three sentences in ChromaDB and search by meaning.
3. **A real wiki page:** cut the EdShopper runbooks into chunks and store them on disk.
4. **RAG:** retrieve the best chunks for an incident question and let the LLM answer from them.

## 1. Install ChromaDB

ChromaDB is the vector database we use. It also brings a small embedding model, so we need no other library for the first part.

In [1]:
%pip install chromadb

  Using cached chromadb-1.5.9-cp39-abi3-macosx_11_0_arm64.whl.metadata (5.0 kB)
  Using cached build-1.6.1-py3-none-any.whl.metadata (5.6 kB)
  Using cached pydantic_settings-2.16.2-py3-none-any.whl.metadata (4.0 kB)
  Using cached pybase64-1.5.2-cp311-cp311-macosx_11_0_arm64.whl.metadata (11 kB)
  Using cached uvicorn-0.54.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached numpy-2.4.6-cp311-cp311-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached onnxruntime-1.31.1-cp311-cp311-macosx_14_0_arm64.whl.metadata (5.8 kB)
  Using cached opentelemetry_api-1.45.2-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.45.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached opentelemetry_sdk-1.45.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached tokenizers-0.23.3-cp310-abi3-macosx_11_0_arm64.whl.metadata (9.8 kB)
  Using cached pypika-0.51.1-py2.py3-none-any.whl.metadata (51 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached

In [2]:
from chromadb.utils import embedding_functions

## 2. Create an embedding function

An embedding function turns text into a vector, which is a list of numbers that captures its meaning.
The default one runs on your laptop, so it needs no API key.

In [3]:
embed = embedding_functions.DefaultEmbeddingFunction()

## 3. Embed three sentences

Two sentences describe the same problem in different words (the pod restarts, the container crashes) and one is unrelated (a cake recipe).
Each sentence becomes one vector. Print it to see the numbers.

In [4]:
sentences = [
    "The pod keeps restarting",
    "The container crashes again and again",
    "Chocolate cake recipe",
]

In [5]:
vectors = embed(sentences)

In [6]:
vectors

[array([ 8.22418481e-02, -7.42845163e-02,  5.88655621e-02, -8.42253771e-03,
         3.80391367e-02, -3.39194573e-02, -2.71251816e-02, -5.56311496e-02,
         5.50322011e-02,  7.92004019e-02, -2.66967621e-02,  1.73828509e-02,
        -1.04694806e-01,  6.54204935e-02, -1.54410182e-02,  2.30938997e-02,
        -2.90528387e-02,  3.26971859e-02, -2.79119965e-02, -4.14673425e-03,
        -1.42809115e-02,  4.44426341e-03,  2.94065587e-02,  6.16287068e-02,
        -5.21903075e-02,  1.72099154e-02, -3.18648890e-02, -5.36339507e-02,
        -2.97433212e-02, -3.13753001e-02, -4.37574610e-02,  3.17425579e-02,
         4.65429537e-02,  9.95071139e-03,  6.35659918e-02,  6.96156695e-02,
        -4.16246168e-02, -3.69702093e-02, -7.00456742e-03, -4.03852155e-03,
        -5.45526110e-02, -8.98067728e-02, -2.93424483e-02, -8.55604419e-04,
         1.66036971e-02,  4.89233918e-02,  4.45160754e-02, -1.54993255e-02,
         1.48449972e-01,  4.29509059e-02, -3.09989844e-02, -6.09945096e-02,
         9.7

## 4. Measure the distance between meanings

Cosine distance compares two vectors: 0 means the same meaning, and a bigger number means further apart.
The pod and container sentences are close (about 0.45). The pod and cake sentences are far (about 0.90), even though the first two share no important words.

In [7]:
from chromadb.utils.distance_functions import cosine

In [8]:
print (cosine (vectors[0], vectors[1]))

0.44666945934295654


In [9]:
print (cosine (vectors[0], vectors[2]))

0.9017191529273987


## 5. Create a collection

A collection is like a table in a database. This client keeps everything in memory, so the data is gone when the notebook stops.
hnsw:space = cosine tells ChromaDB to use cosine distance for the search.

In [10]:
import chromadb

In [11]:
client = chromadb.Client()

In [12]:
collection = client.get_or_create_collection(name="runbook", metadata={"hnsw:space": "cosine"})

## 6. Prepare the data to store

Every item needs an id and the text (the document). Metadata is optional extra information, such as the topic, and you can filter on it later.
The next three cells create these lists.

In [13]:
ids = ["runbook-01", "runbook-02", "runbook-03"]

In [14]:
documents = [
    "The pod keeps restarting",
    "The container crashes again and again",
    "Chocolate cake recipe",
]

In [15]:
metadatas = [
    {"topic": "kubernetes"}, 
    {"topic": "containers"}, 
    {"topic": "recipe"}
]

## 7. Add the items and count them

collection.add() embeds each document and stores the vector together with its text and metadata.
count() should now show 3.

In [16]:
collection.add(ids=ids, documents=documents, metadatas=metadatas)

In [17]:
collection.count()

3

## 8. Look inside the collection

peek() shows the first items. get() returns one item by id, and with embeddings in the include list you also see its vector.
The database keeps the text, the metadata and the vector together.

In [18]:
collection.peek()

{'ids': ['runbook-01', 'runbook-02', 'runbook-03'],
 'embeddings': array([[ 0.08224185, -0.07428452,  0.05886556, ..., -0.00775459,
         -0.02884794, -0.00532245],
        [ 0.07274257,  0.01485308,  0.05459513, ...,  0.0375462 ,
         -0.00239877,  0.02545534],
        [-0.00444834,  0.02947922,  0.02080083, ...,  0.05692742,
         -0.03600193, -0.10591336]], shape=(3, 384)),
 'documents': ['The pod keeps restarting',
  'The container crashes again and again',
  'Chocolate cake recipe'],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [{'topic': 'kubernetes'},
  {'topic': 'containers'},
  {'topic': 'recipe'}]}

In [19]:
item = collection.get(ids=["runbook-01"], include=["documents", "metadatas", "embeddings"])

In [20]:
item

{'ids': ['runbook-01'],
 'embeddings': array([[ 8.22418481e-02, -7.42845163e-02,  5.88655621e-02,
         -8.42253771e-03,  3.80391367e-02, -3.39194573e-02,
         -2.71251816e-02, -5.56311496e-02,  5.50322011e-02,
          7.92004019e-02, -2.66967621e-02,  1.73828509e-02,
         -1.04694806e-01,  6.54204935e-02, -1.54410182e-02,
          2.30938997e-02, -2.90528387e-02,  3.26971859e-02,
         -2.79119965e-02, -4.14673425e-03, -1.42809115e-02,
          4.44426341e-03,  2.94065587e-02,  6.16287068e-02,
         -5.21903075e-02,  1.72099154e-02, -3.18648890e-02,
         -5.36339507e-02, -2.97433212e-02, -3.13753001e-02,
         -4.37574610e-02,  3.17425579e-02,  4.65429537e-02,
          9.95071139e-03,  6.35659918e-02,  6.96156695e-02,
         -4.16246168e-02, -3.69702093e-02, -7.00456742e-03,
         -4.03852155e-03, -5.45526110e-02, -8.98067728e-02,
         -2.93424483e-02, -8.55604419e-04,  1.66036971e-02,
          4.89233918e-02,  4.45160754e-02, -1.54993255e-02,
  

## 9. Search by meaning

The question is about sweets, but no document contains that word. ChromaDB embeds the question and returns the closest documents first.
The cake recipe comes first and the Kubernetes sentences follow. This is semantic search, not keyword search.

In [28]:
question = "I want to know how to prepare my sweets?"

In [29]:
results = collection.query(query_texts=[question], n_results=3)

In [30]:
results

{'ids': [['runbook-03', 'runbook-02', 'runbook-01']],
 'embeddings': None,
 'documents': [['Chocolate cake recipe',
   'The container crashes again and again',
   'The pod keeps restarting']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'topic': 'recipe'},
   {'topic': 'containers'},
   {'topic': 'kubernetes'}]],
 'distances': [[0.5767015218734741, 0.9677658081054688, 0.9706384539604187]]}

## 10. Scenario: a real wiki page

Now we use real content: the EdShopper runbooks page.
Install the text splitter, then read the page into one string.

In [31]:
%pip install langchain-text-splitters

  Using cached langchain_text_splitters-1.1.3-py3-none-any.whl.metadata (3.6 kB)
  Using cached langchain_core-1.8.0-py3-none-any.whl.metadata (4.7 kB)
  Using cached jsonpatch-1.47-py3-none-any.whl.metadata (4.9 kB)
  Using cached langchain_protocol-0.0.21-py3-none-any.whl.metadata (2.4 kB)
  Using cached langsmith-0.14.17-py3-none-any.whl.metadata (39 kB)
  Using cached uuid_utils-1.2.0-cp310-abi3-macosx_10_12_x86_64.macosx_11_0_arm64.macosx_10_12_universal2.whl.metadata (6.4 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached uuid_utils-0.19.0-cp310-abi3-macosx_10_12_x86_64.macosx_11_0_arm64.macosx_10_12_universal2.whl.metadata (6.4 kB)
  Using cached xxhash-4.1.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (18 kB)
  Using cached zstandard-0.25.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (3.3 kB)
Using cached langchain_text_splitters-1.1.3-py3-none-any.whl (38 kB)
Using cached 

In [33]:
file = "wiki_pages/edshopper-runbooks.md"

In [34]:
text = open(file, encoding="utf-8").read()

In [35]:
text

'# EdShopper runbooks: troubleshooting steps\n- Space: Runbooks\n- Labels: runbook, troubleshooting, kubernetes, crashloopbackoff, oomkilled, imagepullbackoff, rollback, 5xx, database, backup, tls, dns, terraform\n- Owner: sre-team\n- Last updated: 2026-11-10\n\n## How to read these runbooks\nThe commands use the service checkout-api in the namespace checkout as the example. For another service, use its namespace and deployment name from the page "EdShopper applications, owners and on-call contacts". Check that kubectl points to the right cluster first: `kubectl config current-context` (edshopper-prod, edshopper-stage or edshopper-dev). Alerts arrive in #alerts-prod, #alerts-stage and #alerts-dev.\n\n## Pod in CrashLoopBackOff: symptoms and causes\nA pod shows the status CrashLoopBackOff and its restart count keeps growing. The alert KubePodCrashLooping appears in #alerts-prod. Users may see errors because fewer pods are serving traffic.\nLikely causes:\n- The application exits at star

## 11. Cut the page into chunks

A whole page is too big to search and to send to the LLM, so we cut it into pieces. RecursiveCharacterTextSplitter cuts at paragraphs and lines first, with chunk_size=500 characters and chunk_overlap=50, so a sentence at the border is not lost.
The runbooks page gives 43 chunks.

In [36]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [37]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

In [38]:
splitter

In [39]:
chunks = splitter.split_text(text)

In [40]:
chunks

['# EdShopper runbooks: troubleshooting steps\n- Space: Runbooks\n- Labels: runbook, troubleshooting, kubernetes, crashloopbackoff, oomkilled, imagepullbackoff, rollback, 5xx, database, backup, tls, dns, terraform\n- Owner: sre-team\n- Last updated: 2026-11-10',
 '## How to read these runbooks\nThe commands use the service checkout-api in the namespace checkout as the example. For another service, use its namespace and deployment name from the page "EdShopper applications, owners and on-call contacts". Check that kubectl points to the right cluster first: `kubectl config current-context` (edshopper-prod, edshopper-stage or edshopper-dev). Alerts arrive in #alerts-prod, #alerts-stage and #alerts-dev.',
 '## Pod in CrashLoopBackOff: symptoms and causes\nA pod shows the status CrashLoopBackOff and its restart count keeps growing. The alert KubePodCrashLooping appears in #alerts-prod. Users may see errors because fewer pods are serving traffic.\nLikely causes:\n- The application exits at s

In [41]:
len(chunks)

43

## 12. Put the page title in front of each chunk

A chunk alone does not say where it comes from. We add the title in front of every chunk, so the text we store, and later give to the LLM, always carries its source.

In [43]:
title = "edshopper rubook"

In [44]:
documents = []
for chunk in chunks:
    documents.append(title + "\n" + chunk)

print(documents)

['edshopper rubook\n# EdShopper runbooks: troubleshooting steps\n- Space: Runbooks\n- Labels: runbook, troubleshooting, kubernetes, crashloopbackoff, oomkilled, imagepullbackoff, rollback, 5xx, database, backup, tls, dns, terraform\n- Owner: sre-team\n- Last updated: 2026-11-10', 'edshopper rubook\n## How to read these runbooks\nThe commands use the service checkout-api in the namespace checkout as the example. For another service, use its namespace and deployment name from the page "EdShopper applications, owners and on-call contacts". Check that kubectl points to the right cluster first: `kubectl config current-context` (edshopper-prod, edshopper-stage or edshopper-dev). Alerts arrive in #alerts-prod, #alerts-stage and #alerts-dev.', 'edshopper rubook\n## Pod in CrashLoopBackOff: symptoms and causes\nA pod shows the status CrashLoopBackOff and its restart count keeps growing. The alert KubePodCrashLooping appears in #alerts-prod. Users may see errors because fewer pods are serving tr

In [45]:
import chromadb

## 13. Create a persistent collection

PersistentClient saves the data in the folder chroma_db, so it is still there after a restart. The new collection runbookdata starts empty.
create_collection gives an error if the collection already exists, so delete the folder chroma_db before you run this cell again.

In [46]:
client = chromadb.PersistentClient(path="chroma_db")

In [47]:
collection = client.create_collection(name="runbookdata", metadata={"hnsw:space": "cosine"})

In [48]:
print("Collection ready. Items inside:", collection.count())

Collection ready. Items inside: 0


In [49]:
len(chunks)

43

## 14. Build ids and metadata, then store the chunks

Each chunk gets a unique id (file name plus a number) and metadata with the file and the title.
After collection.add() the count shows 43 items, one per chunk.

In [51]:
ids = []
metadatas = []
for number in range(len(documents)):
    ids.append(f"{file}-{number + 1}")
    metadatas.append({"file": file, "title": title})

In [52]:
ids

['wiki_pages/edshopper-runbooks.md-1',
 'wiki_pages/edshopper-runbooks.md-2',
 'wiki_pages/edshopper-runbooks.md-3',
 'wiki_pages/edshopper-runbooks.md-4',
 'wiki_pages/edshopper-runbooks.md-5',
 'wiki_pages/edshopper-runbooks.md-6',
 'wiki_pages/edshopper-runbooks.md-7',
 'wiki_pages/edshopper-runbooks.md-8',
 'wiki_pages/edshopper-runbooks.md-9',
 'wiki_pages/edshopper-runbooks.md-10',
 'wiki_pages/edshopper-runbooks.md-11',
 'wiki_pages/edshopper-runbooks.md-12',
 'wiki_pages/edshopper-runbooks.md-13',
 'wiki_pages/edshopper-runbooks.md-14',
 'wiki_pages/edshopper-runbooks.md-15',
 'wiki_pages/edshopper-runbooks.md-16',
 'wiki_pages/edshopper-runbooks.md-17',
 'wiki_pages/edshopper-runbooks.md-18',
 'wiki_pages/edshopper-runbooks.md-19',
 'wiki_pages/edshopper-runbooks.md-20',
 'wiki_pages/edshopper-runbooks.md-21',
 'wiki_pages/edshopper-runbooks.md-22',
 'wiki_pages/edshopper-runbooks.md-23',
 'wiki_pages/edshopper-runbooks.md-24',
 'wiki_pages/edshopper-runbooks.md-25',
 'wiki_pa

In [53]:
metadatas

[{'file': 'wiki_pages/edshopper-runbooks.md', 'title': 'edshopper rubook'},
 {'file': 'wiki_pages/edshopper-runbooks.md', 'title': 'edshopper rubook'},
 {'file': 'wiki_pages/edshopper-runbooks.md', 'title': 'edshopper rubook'},
 {'file': 'wiki_pages/edshopper-runbooks.md', 'title': 'edshopper rubook'},
 {'file': 'wiki_pages/edshopper-runbooks.md', 'title': 'edshopper rubook'},
 {'file': 'wiki_pages/edshopper-runbooks.md', 'title': 'edshopper rubook'},
 {'file': 'wiki_pages/edshopper-runbooks.md', 'title': 'edshopper rubook'},
 {'file': 'wiki_pages/edshopper-runbooks.md', 'title': 'edshopper rubook'},
 {'file': 'wiki_pages/edshopper-runbooks.md', 'title': 'edshopper rubook'},
 {'file': 'wiki_pages/edshopper-runbooks.md', 'title': 'edshopper rubook'},
 {'file': 'wiki_pages/edshopper-runbooks.md', 'title': 'edshopper rubook'},
 {'file': 'wiki_pages/edshopper-runbooks.md', 'title': 'edshopper rubook'},
 {'file': 'wiki_pages/edshopper-runbooks.md', 'title': 'edshopper rubook'},
 {'file': 'w

In [54]:
collection.add(ids=ids, documents=documents, metadatas=metadatas)

In [55]:
print("Collection ready. Items inside:", collection.count())

Collection ready. Items inside: 43


## 15. Search the wiki

We ask an incident question and get back the best chunk: the CrashLoopBackOff diagnosis runbook, although the question uses other words.
The wiki is now searchable by meaning.

In [56]:
results = collection.query(query_texts=["Pods keep restarting. What should I check first?"], n_results=1)

In [57]:
results

{'ids': [['wiki_pages/edshopper-runbooks.md-5']],
 'embeddings': None,
 'documents': [['edshopper rubook\n## Pod in CrashLoopBackOff: diagnosis and fix\n1. List the pods and their restarts: `kubectl get pods -n checkout`\n2. Look at the reason and the exit code: `kubectl describe pod checkout-api-6d8f7c9b5-x2k4p -n checkout` and read "Last State".\n3. Read the log of the container that crashed: `kubectl logs checkout-api-6d8f7c9b5-x2k4p -n checkout --previous`\n4. Exit code 137 means OOMKilled. Exit code 1 means the application raised an error: read the last lines of the log.\nFix:']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'file': 'wiki_pages/edshopper-runbooks.md',
    'title': 'edshopper rubook'}]],
 'distances': [[0.35126930475234985]]}

## 16. Scenario: answer with RAG

RAG means Retrieval-Augmented Generation: first retrieve the right chunks, then let the LLM answer only from them.
The LLM runs on Groq, so we paste the API key in a hidden box (getpass). It stays in memory and is never saved in the notebook.

In [58]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Paste your Groq API key: ")

Paste your Groq API key:  ········


## 17. Retrieve the top 4 chunks

We ask for 4 results instead of 1, to give the LLM more context.
The loop prints the distance and the heading of each chunk: smaller is closer.

In [59]:
question = "Pods keep restarting. What should I check first?"

results = collection.query(query_texts=[question], n_results=4)
results

{'ids': [['wiki_pages/edshopper-runbooks.md-5',
   'wiki_pages/edshopper-runbooks.md-3',
   'wiki_pages/edshopper-runbooks.md-7',
   'wiki_pages/edshopper-runbooks.md-9']],
 'embeddings': None,
 'documents': [['edshopper rubook\n## Pod in CrashLoopBackOff: diagnosis and fix\n1. List the pods and their restarts: `kubectl get pods -n checkout`\n2. Look at the reason and the exit code: `kubectl describe pod checkout-api-6d8f7c9b5-x2k4p -n checkout` and read "Last State".\n3. Read the log of the container that crashed: `kubectl logs checkout-api-6d8f7c9b5-x2k4p -n checkout --previous`\n4. Exit code 137 means OOMKilled. Exit code 1 means the application raised an error: read the last lines of the log.\nFix:',
   'edshopper rubook\n## Pod in CrashLoopBackOff: symptoms and causes\nA pod shows the status CrashLoopBackOff and its restart count keeps growing. The alert KubePodCrashLooping appears in #alerts-prod. Users may see errors because fewer pods are serving traffic.\nLikely causes:\n- The

In [61]:
for i in range(4):
    meta = results["metadatas"][0][i]
    distance = round(results["distances"][0][i], 2)
    first_line = results["documents"][0][i].splitlines()[1][:60]     # the first line after the title
    print(distance, "|", first_line)

0.35 | ## Pod in CrashLoopBackOff: diagnosis and fix
0.35 | ## Pod in CrashLoopBackOff: symptoms and causes
0.36 | ## Container OOMKilled: symptoms and causes
0.42 | ## Container OOMKilled: diagnosis and fix


## 18. Write the system prompt

The system prompt gives the LLM its role and its rules: use only the wiki pages, give numbered steps with the exact commands, and say "I could not find this in the wiki" when the answer is missing.
These rules keep the answer grounded in your runbooks.

In [62]:
system_prompt = """You are the Platform Operations Assistant. You help an operations team with incidents
and platform questions, using the wiki pages that are given to you.
Answer the question using ONLY these wiki pages.

Rules:
- For an incident or a problem: give short, numbered steps and include the exact commands from the pages.
- For a question about servers, databases or configuration: give the exact value from the pages.
- If the page says an action needs approval, say so clearly.
- If the pages do not contain the answer, say: I could not find this in the wiki.
- Never invent commands, numbers or names.
- Finish with one line: Source: <page title>"""

## 19. Build the user prompt

The retrieved chunks become the context, and they go into the user prompt together with the question.
Print user_prompt to see exactly what the LLM will receive.

In [64]:
context = results['documents']
context

[['edshopper rubook\n## Pod in CrashLoopBackOff: diagnosis and fix\n1. List the pods and their restarts: `kubectl get pods -n checkout`\n2. Look at the reason and the exit code: `kubectl describe pod checkout-api-6d8f7c9b5-x2k4p -n checkout` and read "Last State".\n3. Read the log of the container that crashed: `kubectl logs checkout-api-6d8f7c9b5-x2k4p -n checkout --previous`\n4. Exit code 137 means OOMKilled. Exit code 1 means the application raised an error: read the last lines of the log.\nFix:',
  'edshopper rubook\n## Pod in CrashLoopBackOff: symptoms and causes\nA pod shows the status CrashLoopBackOff and its restart count keeps growing. The alert KubePodCrashLooping appears in #alerts-prod. Users may see errors because fewer pods are serving traffic.\nLikely causes:\n- The application exits at startup because an environment variable, ConfigMap or Secret is missing or wrong.\n- The liveness probe fails, so Kubernetes keeps killing a healthy but slow container.',
  'edshopper rub

In [65]:
user_prompt = f"""Wiki pages:
{context}
Question: {question}"""

In [66]:
user_prompt

'Wiki pages:\n[[\'edshopper rubook\\n## Pod in CrashLoopBackOff: diagnosis and fix\\n1. List the pods and their restarts: `kubectl get pods -n checkout`\\n2. Look at the reason and the exit code: `kubectl describe pod checkout-api-6d8f7c9b5-x2k4p -n checkout` and read "Last State".\\n3. Read the log of the container that crashed: `kubectl logs checkout-api-6d8f7c9b5-x2k4p -n checkout --previous`\\n4. Exit code 137 means OOMKilled. Exit code 1 means the application raised an error: read the last lines of the log.\\nFix:\', \'edshopper rubook\\n## Pod in CrashLoopBackOff: symptoms and causes\\nA pod shows the status CrashLoopBackOff and its restart count keeps growing. The alert KubePodCrashLooping appears in #alerts-prod. Users may see errors because fewer pods are serving traffic.\\nLikely causes:\\n- The application exits at startup because an environment variable, ConfigMap or Secret is missing or wrong.\\n- The liveness probe fails, so Kubernetes keeps killing a healthy but slow con

## 20. Send the messages to the LLM

messages holds the system and the user roles. We create the client for Groq's OpenAI-compatible API and call the model with temperature=0 for steady answers.
openai/gpt-oss-120b is a reasoning model, so max_tokens=2000 leaves room for its thinking.

In [67]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

In [68]:
from openai import OpenAI

In [69]:
llm = OpenAI(api_key=os.environ["GROQ_API_KEY"], base_url="https://api.groq.com/openai/v1")
MODEL = "openai/gpt-oss-120b"

In [70]:
response = llm.chat.completions.create(
    model=MODEL,
    messages=messages,
    temperature=0,        # low = steady, factual answers
    max_tokens=2000,
)

## 21. Read the answer

The full response has metadata, and the text is in response.choices[0].message.content.
The answer gives numbered steps with the kubectl commands and ends with a source line, so you can check it against the wiki.

In [71]:
response

ChatCompletion(id='chatcmpl-77429cbe-f374-45c9-ae15-7ef4ec5c661a', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='1. List the pods and see their restart counts:  \n   ```bash\n   kubectl get pods -n checkout\n   ```  \n\nSource: edshopper rubook – Pod in CrashLoopBackOff: diagnosis and fix', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='We need to answer using only the wiki pages. The question: "Pods keep restarting. What should I check first?" Likely from "Pod in CrashLoopBackOff: diagnosis and fix" page. The first step is "List the pods and their restarts: `kubectl get pods -n checkout`". So answer: check pod restarts list. Provide short numbered steps. So step 1: list pods and restarts. Provide command. Source page title: "edshopper rubook\\n## Pod in CrashLoopBackOff: diagnosis and fix".'))], created=1794453494, model='openai/gpt-oss-120b', object='chat.completion',

In [72]:
print(response.choices[0].message.content)

1. List the pods and see their restart counts:  
   ```bash
   kubectl get pods -n checkout
   ```  

Source: edshopper rubook – Pod in CrashLoopBackOff: diagnosis and fix
